# Basic Grid Object Detector (PyTorch, ReLU, Adam)

Tiny YOLO-style CNN: the image is split into a `GRID x GRID` grid; each cell predicts
objectness + one box `(cx, cy, w, h)`. Single class.

**Colab setup**
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload `dataset_10pct.zip` (preferred, much faster) **or** the `dataset_10pct/` folder to the root of *My Drive*.
3. Run all cells. A resized image cache and the best checkpoint are saved to `My Drive/5810_detector/`, so later runs skip preprocessing.

In [ ]:
import os, sys, time, glob, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/dataset_10pct"# folder (or DATA_DIR + ".zip") with train/ valid/ test/
    OUT_DIR = "/content/drive/MyDrive/5810_detector"  # cache + checkpoints persist here
else:
    DATA_DIR = "dataset_10pct"
    OUT_DIR = "."
os.makedirs(OUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)

In [ ]:
# ---- Config ----
IMG_SIZE = 160            # source images are 640x640; 160 = exact 1/4 JPEG downscale
GRID = IMG_SIZE // 16     # 4 stride-2 pools -> 10x10 grid
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-3
TIME_LIMIT_MIN = 20       # stop training early if this is exceeded
BOX_WEIGHT = 5.0          # box loss weight (YOLOv1-style)
NOOBJ_WEIGHT = 0.5        # weight of empty-cell objectness loss
CONF_THRESH = 0.5         # score threshold for precision/recall + visualization
NMS_IOU = 0.4
torch.manual_seed(0)

## Data: load once, resize, cache as uint8 tensors

In [ ]:
CACHE = f"{OUT_DIR}/cache_{IMG_SIZE}.pt"

EXTRACT_DIR = "/content/data" if IN_COLAB else "/tmp/data"

def find_root(base):
    """Folder under base containing train/images (works with or without a top-level folder)."""
    hits = sorted(glob.glob(f"{base}/**/train/images", recursive=True), key=len)
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None

def resolve_data_dir():
    zip_path = DATA_DIR + ".zip"
    if os.path.exists(zip_path):  # prefer zip: 1 Drive read + local disk beats ~2k Drive file reads
        if not find_root(EXTRACT_DIR):
            print("Extracting", zip_path)
            zipfile.ZipFile(zip_path).extractall(EXTRACT_DIR)
        root = find_root(EXTRACT_DIR)
    else:
        root = find_root(DATA_DIR) if os.path.isdir(DATA_DIR) else None
    parent = os.path.dirname(os.path.abspath(DATA_DIR))
    assert root, f"No train/images found in {DATA_DIR} or {zip_path}. Contents of {parent}: {sorted(os.listdir(parent))}"
    print("Using data from", root)
    return root

def load_image(path):
    img = Image.open(path)
    img.draft("RGB", (IMG_SIZE, IMG_SIZE))  # fast downscale during JPEG decode
    img = img.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    return torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1)

def load_labels(path):
    rows = []
    if os.path.exists(path):
        rows = [[float(v) for v in line.split()[1:5]] for line in open(path) if line.strip()]
    return torch.tensor(rows, dtype=torch.float32).view(-1, 4)  # (n, 4) cx, cy, w, h in [0, 1]

def load_split(root, split):
    img_dir = f"{root}/{split}/images"
    paths = sorted(p for p in glob.glob(f"{img_dir}/*") if p.lower().endswith((".jpg", ".jpeg", ".png")))
    assert paths, f"No images in {img_dir} (contains: {os.listdir(img_dir)[:5]}). Is the Drive upload finished?"
    label_paths = [f"{root}/{split}/labels/{os.path.splitext(os.path.basename(p))[0]}.txt" for p in paths]
    with ThreadPoolExecutor(16) as ex:
        images = list(ex.map(load_image, paths))
        boxes = list(ex.map(load_labels, label_paths))
    return torch.stack(images), boxes

if os.path.exists(CACHE):
    data = torch.load(CACHE)
    print("Loaded cache", CACHE)
else:
    root = resolve_data_dir()
    data = {}
    for split in ["train", "valid", "test"]:
        t0 = time.time()
        data[split] = load_split(root, split)
        print(f"{split}: {len(data[split][1])} images in {time.time() - t0:.0f}s")
    torch.save(data, CACHE)

def build_targets(boxes_list):
    """(N, 5, GRID, GRID): objectness, x/y offset within cell, w, h."""
    t = torch.zeros(len(boxes_list), 5, GRID, GRID)
    for i, boxes in enumerate(boxes_list):
        for cx, cy, w, h in boxes.tolist():
            gx, gy = min(int(cx * GRID), GRID - 1), min(int(cy * GRID), GRID - 1)
            t[i, :, gy, gx] = torch.tensor([1.0, cx * GRID - gx, cy * GRID - gy, w, h])
    return t

# Whole dataset fits in GPU memory, so skip DataLoader entirely
splits = {}
for split, (imgs, boxes) in data.items():
    splits[split] = (imgs.to(device), build_targets(boxes).to(device), boxes)
    n_obj = sum(len(b) for b in boxes)
    print(f"{split}: {len(boxes)} images, {n_obj} objects")

## Model

In [ ]:
def conv_block(c_in, c_out, pool=True):
    layers = [nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True)]
    if pool:
        layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)

class GridDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            conv_block(3, 32),     # 160 -> 80
            conv_block(32, 64),    # 80 -> 40
            conv_block(64, 128),   # 40 -> 20
            conv_block(128, 256),  # 20 -> 10
            conv_block(256, 256, pool=False),
        )
        self.head = nn.Conv2d(256, 5, 1)  # per cell: obj logit, x, y, w, h (raw)

    def forward(self, x):
        return self.head(self.backbone(x))

def normalize(imgs_uint8):
    return imgs_uint8.float() / 255.0

model = GridDetector().to(device)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

## Loss, decoding, metrics

In [ ]:
def detection_loss(pred, t):
    mask = t[:, 0]
    bce = F.binary_cross_entropy_with_logits(pred[:, 0], mask, reduction="none")
    n_pos = mask.sum().clamp(min=1)
    obj_loss = ((bce * mask).sum() + NOOBJ_WEIGHT * (bce * (1 - mask)).sum()) / n_pos
    xy, wh = torch.sigmoid(pred[:, 1:3]), torch.sigmoid(pred[:, 3:5])
    box_err = ((xy - t[:, 1:3]) ** 2).sum(1) + ((wh.sqrt() - t[:, 3:5].sqrt()) ** 2).sum(1)
    box_loss = (box_err * mask).sum() / n_pos
    return obj_loss + BOX_WEIGHT * box_loss

def to_xyxy(b):
    return torch.cat([b[:, :2] - b[:, 2:] / 2, b[:, :2] + b[:, 2:] / 2], 1)

def pairwise_iou(a, b):
    """(N, 4) x (M, 4) cx,cy,w,h -> (N, M) IoU."""
    a, b = to_xyxy(a), to_xyxy(b)
    lt = torch.max(a[:, None, :2], b[None, :, :2])
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])
    inter = (rb - lt).clamp(min=0).prod(2)
    area_a = (a[:, 2:] - a[:, :2]).prod(1)
    area_b = (b[:, 2:] - b[:, :2]).prod(1)
    return inter / (area_a[:, None] + area_b[None] - inter).clamp(min=1e-9)

def nms(boxes, scores, iou_thresh):
    order, keep = scores.argsort(descending=True), []
    while order.numel():
        i = order[0]
        keep.append(i.item())
        ious = pairwise_iou(boxes[i:i + 1], boxes[order[1:]])[0]
        order = order[1:][ious <= iou_thresh]
    return torch.tensor(keep, dtype=torch.long)

def decode(pred, conf_thresh):
    """Raw (B, 5, GRID, GRID) output -> list of (boxes (n, 4), scores (n,)) per image."""
    pred = pred.float().cpu()
    gy, gx = torch.meshgrid(torch.arange(GRID), torch.arange(GRID), indexing="ij")
    conf = torch.sigmoid(pred[:, 0])
    s = torch.sigmoid(pred[:, 1:])
    boxes = torch.stack([(gx + s[:, 0]) / GRID, (gy + s[:, 1]) / GRID, s[:, 2], s[:, 3]], -1)
    out = []
    for i in range(len(pred)):
        keep = conf[i] > conf_thresh
        b, sc = boxes[i][keep], conf[i][keep]
        k = nms(b, sc, NMS_IOU)
        out.append((b[k], sc[k]))
    return out

def detection_metrics(preds, gts, iou_thresh=0.5):
    """AP@0.5 (all-point interpolation) + precision/recall at CONF_THRESH."""
    records, n_gt = [], sum(len(g) for g in gts)
    for (boxes, scores), gt in zip(preds, gts):
        matched = torch.zeros(len(gt), dtype=torch.bool)
        ious = pairwise_iou(boxes, gt) if len(boxes) and len(gt) else None
        for j in scores.argsort(descending=True).tolist():
            tp = False
            if ious is not None:
                iou, k = ious[j].masked_fill(matched, -1).max(0)
                if iou >= iou_thresh:
                    matched[k], tp = True, True
            records.append((scores[j].item(), tp))
    if not records:
        return {"AP50": 0.0, "P": 0.0, "R": 0.0}
    records.sort(key=lambda r: -r[0])
    scores = torch.tensor([r[0] for r in records])
    tp = torch.tensor([r[1] for r in records], dtype=torch.float32)
    cum_tp = tp.cumsum(0)
    recall = cum_tp / max(n_gt, 1)
    precision = cum_tp / torch.arange(1, len(tp) + 1)
    envelope = precision.flip(0).cummax(0).values.flip(0)
    ap = ((recall - torch.cat([torch.zeros(1), recall[:-1]])) * envelope).sum().item()
    at = scores >= CONF_THRESH
    p = tp[at].sum().item() / max(at.sum().item(), 1)
    r = tp[at].sum().item() / max(n_gt, 1)
    return {"AP50": ap, "P": p, "R": r}

@torch.no_grad()
def evaluate(model, split, batch_size=128):
    imgs, targets, gts = splits[split]
    model.eval()
    preds, loss_sum = [], 0.0
    for i in range(0, len(imgs), batch_size):
        out = model(normalize(imgs[i:i + batch_size]))
        loss_sum += detection_loss(out, targets[i:i + batch_size]).item() * len(out)
        preds += decode(out, conf_thresh=0.05)
    return {"loss": loss_sum / len(imgs), **detection_metrics(preds, gts)}, preds

## Train

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
train_imgs, train_targets, _ = splits["train"]
best_path = f"{OUT_DIR}/detector_best.pt"
best_ap, start = -1.0, time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(train_imgs), device=device)
    loss_sum = 0.0
    for i in range(0, len(perm), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        loss = detection_loss(model(normalize(train_imgs[idx])), train_targets[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(idx)

    val, _ = evaluate(model, "valid")
    elapsed = (time.time() - start) / 60
    print(f"Epoch {epoch:02d} | train loss {loss_sum / len(perm):.4f} | val loss {val['loss']:.4f} "
          f"AP50 {val['AP50']:.3f} P {val['P']:.3f} R {val['R']:.3f} | {elapsed:.1f} min")
    if val["AP50"] > best_ap:
        best_ap = val["AP50"]
        torch.save(model.state_dict(), best_path)
    if elapsed > TIME_LIMIT_MIN:
        print(f"Time limit of {TIME_LIMIT_MIN} min reached")
        break

print(f"Best val AP50: {best_ap:.3f} -> {best_path}")

## Test set evaluation

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
test, test_preds = evaluate(model, "test")
print(f"Test | loss {test['loss']:.4f} AP50 {test['AP50']:.3f} P {test['P']:.3f} R {test['R']:.3f}")

## Visualize predictions (green = ground truth, red = prediction)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def draw(ax, boxes, color):
    for cx, cy, w, h in boxes.tolist():
        ax.add_patch(Rectangle(((cx - w / 2) * IMG_SIZE, (cy - h / 2) * IMG_SIZE), w * IMG_SIZE, h * IMG_SIZE,
                               fill=False, edgecolor=color, linewidth=1.5))

test_imgs, _, test_gts = splits["test"]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.flat, torch.randperm(len(test_gts))[:8].tolist()):
    ax.imshow(test_imgs[i].permute(1, 2, 0).cpu())
    boxes, scores = test_preds[i]
    draw(ax, test_gts[i], "lime")
    draw(ax, boxes[scores >= CONF_THRESH], "red")
    ax.axis("off")
plt.tight_layout()
plt.show()